# Hanabi - Data Engineering

### Game data preparation

The data is obtained from https://github.com/yawgmoth/HanabiData.

In [1]:
import re
import random
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
import zipfile
import io
import random

The repo is 9 years old and contains code made with Python 2. They included the following code to create the deck with a random seed given in the game logs:

In [4]:
# def make_deck(seed):
#     random.seed(seed)
#     deck = []
#     for col in ["green", "yellow", "white", "blue", "red"]:
#         for num, cnt in enumerate([3,2,2,2,1]):
#             for i in xrange(cnt):
#                 deck.append((col, num+1))
#     random.shuffle(deck)
#     return deck

We had to change the logic to Python 3 and used GenAI to create a function which simulates the Python 2 shuffle:

In [2]:
def py2_shuffle(items):
    for i in range(len(items) - 1, 0, -1):
        j = int(random.random() * (i + 1))
        items[i], items[j] = items[j], items[i]

def make_deck(seed):
    colors = ["green", "yellow", "white", "blue", "red"]
    random.seed(seed)
    deck = []
    for col in colors:
        for num, cnt in enumerate([3, 2, 2, 2, 1]):
            for i in range(cnt):
                deck.append((col, num + 1))
    
    py2_shuffle(deck)
    return deck

This function extracts the start cards for both players by using the make_deck function:

In [3]:
def process_hanabi_log(log_content, filename):
    lines = log_content.split('\n')

    seed = None
    final_score = None
    first_player = None

    for line in lines:
        line = line.strip()

        if line.startswith("Treatment:"): #after the treatment, we can find the seed for the deck
            match = re.search(r",\s*(\d+)\)", line)
            if match:
                seed = int(match.group(1)) #the seed used for the make_deck function

        elif line.startswith("MOVE:") and first_player is None: #defines the first player
            first_player = int(line.split()[1]) #0 is the AI, 1 is the human player

        elif line.startswith("Score"):
            final_score = int(line.split()[-1]) #the final score is found at the end of the line after "Score:"

    if seed is None or first_player is None:
        return None  # Incomplete file skipped

    deck = make_deck(seed)

    if first_player == 0: #0 is the AI
        p1_cards, p2_cards = deck[0:5], deck[5:10] # p1 is the AI
    else: #1 is the human player
        p2_cards, p1_cards = deck[0:5], deck[5:10] # p2 is the human player

  #one hot encoding
    colors = ["green", "yellow", "white", "blue", "red"]
    row_data = {"file_source": filename, "final_score": final_score}
    
    for player_prefix, hand in [("p1", p1_cards), ("p2", p2_cards)]:
        for col in colors:
            for num in range(1, 6):
                col_name = f"{player_prefix}_{col}{num}"
                row_data[col_name] = hand.count((col, num)) #counts how many times the card appears
                
    return row_data

Download the data from the GitHub Repo.

In [4]:
def download_repo_zip():
    """loading the whole repo as zip to the RAM (1 Request)."""
    url = "https://github.com/yawgmoth/HanabiData/archive/refs/heads/master.zip"
    print("Loading repository as ZIP...")
    response = requests.get(url, timeout=120)
    response.raise_for_status()
    print(f"Download finished ({len(response.content) / 1_000_000:.1f} MB).")
    return zipfile.ZipFile(io.BytesIO(response.content))


def build_dataset(output_csv="hanabi_dataset_raw.csv"):
    """Process all .log files and save the result as a CSV."""
    zf = download_repo_zip()

    # Filter all .log files in the log/ directory
    log_files = [
        f for f in zf.namelist()
        if "/log/" in f and f.endswith(".log")
    ]
    print(f"{len(log_files)} Log files found.\n")

    rows = []
    errors = []

    for i, filepath in enumerate(log_files):
        filename = filepath.split("/")[-1]
        try:
            with zf.open(filepath) as f:
                content = f.read().decode("utf-8")
            row = process_hanabi_log(content, filename)
            if row is not None:
                rows.append(row)
        except Exception as e:
            errors.append((filename, str(e)))

        # Progression
        if (i + 1) % 200 == 0:
            print(f"  {i + 1}/{len(log_files)} processed...")

    print(f"\nDone: {len(rows)} rows extracted, {len(errors)} Errors.")

    if errors:
        print("Corrupted files:")
        for name, err in errors[:10]:  # show a maximum of 10
            print(f"  {name}: {err}")

    df = pd.DataFrame(rows)
    df.to_csv(output_csv, index=False)
    print(f"\nDataset saved as: {output_csv}")
    print(df.head())
    return df

Create the csv-file.

In [54]:
df = build_dataset()
df.head

Loading repository as ZIP...
Download finished (2.6 MB).
2280 Log files found.

  200/2280 processed...
  400/2280 processed...
  600/2280 processed...
  800/2280 processed...
  1000/2280 processed...
  1200/2280 processed...
  1400/2280 processed...
  1600/2280 processed...
  1800/2280 processed...
  2000/2280 processed...
  2200/2280 processed...

Done: 2040 rows extracted, 0 Errors.

Dataset saved as: hanabi_dataset_raw.csv
                file_source  final_score  p1_green1  p1_green2  p1_green3  p1_green4  p1_green5  p1_yellow1  p1_yellow2  p1_yellow3  p1_yellow4  p1_yellow5  p1_white1  p1_white2  p1_white3  p1_white4  p1_white5  p1_blue1  p1_blue2  p1_blue3  p1_blue4  p1_blue5  p1_red1  p1_red2  p1_red3  p1_red4  p1_red5  p2_green1  p2_green2  p2_green3  p2_green4  p2_green5  p2_yellow1  p2_yellow2  p2_yellow3  p2_yellow4  p2_yellow5  p2_white1  p2_white2  p2_white3  p2_white4  p2_white5  p2_blue1  p2_blue2  p2_blue3  p2_blue4  p2_blue5  p2_red1  p2_red2  p2_red3  p2_red4  p2_red

<bound method NDFrame.head of                    file_source  final_score  p1_green1  p1_green2  p1_green3  p1_green4  p1_green5  p1_yellow1  p1_yellow2  p1_yellow3  p1_yellow4  p1_yellow5  p1_white1  p1_white2  p1_white3  p1_white4  p1_white5  p1_blue1  p1_blue2  p1_blue3  p1_blue4  p1_blue5  p1_red1  p1_red2  p1_red3  p1_red4  p1_red5  p2_green1  p2_green2  p2_green3  p2_green4  p2_green5  p2_yellow1  p2_yellow2  p2_yellow3  p2_yellow4  p2_yellow5  p2_white1  p2_white2  p2_white3  p2_white4  p2_white5  p2_blue1  p2_blue2  p2_blue3  p2_blue4  p2_blue5  p2_red1  p2_red2  p2_red3  p2_red4  p2_red5
0     game003d9bcb9d27dacf.log           15          1          0          0          0          0           1           1           0           0           0          0          0          0          0          0         0         0         0         0         1        1        0        0        0        0          0          0          0          1          0           0           0         

Check the first few rows of the dataset.

# Clean up

In [55]:
hand_cols = [c for c in df.columns if c not in ["file_source", "final_score", "p1_hand_sum", "p2_hand_sum", "p1_total_cards", "p2_total_cards", "p1_avg_value", "p2_avg_value"]]

original_len = len(df)

keep_indices = (
    df.groupby(hand_cols)
      .apply(
          lambda g: (
              g.sample(2, random_state=42).index
              if len(g) > 3
              else g.index
          )
      )
      .explode()
      .astype(int)
)

df = df.loc[keep_indices].reset_index(drop=True)

print("Original rows:", original_len)
print("Cleaned rows:", len(df))
print("Original columns:", df.shape[1])
print("Cleaned columns:", df.shape[1])

state_frequencies = (
    df[hand_cols]
    .value_counts()
    .sort_values(ascending=False)
)

labels = []
for state in state_frequencies.index:
    active_features = [
        col
        for col, value in zip(hand_cols, state)
        if value != 0
    ]

    labels.append(", ".join(active_features) if active_features else "None")

df.head()

Original rows: 2040
Cleaned rows: 1810
Original columns: 52
Cleaned columns: 52


,file_source,final_score,p1_green1,p1_green2,p1_green3,p1_green4,p1_green5,p1_yellow1,p1_yellow2,p1_yellow3,p1_yellow4,p1_yellow5,p1_white1,p1_white2,p1_white3,p1_white4,p1_white5,p1_blue1,p1_blue2,p1_blue3,p1_blue4,p1_blue5,p1_red1,p1_red2,p1_red3,p1_red4,p1_red5,p2_green1,p2_green2,p2_green3,p2_green4,p2_green5,p2_yellow1,p2_yellow2,p2_yellow3,p2_yellow4,p2_yellow5,p2_white1,p2_white2,p2_white3,p2_white4,p2_white5,p2_blue1,p2_blue2,p2_blue3,p2_blue4,p2_blue5,p2_red1,p2_red2,p2_red3,p2_red4,p2_red5
0,gamec4a276e8bbb31378.log,10,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,2,1,0,1,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,0
1,game146052cdf75868b1.log,21,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,1,1,0,0,1,0,0,0,0,0,0,1,0,0,0,1,1,0,0,0,1,0,0,0,0,0,0,0,1,0
2,gamea89f449023dbd46f.log,8,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,2,0,1,0,0,0,1,0,0,0,0,0,2,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0
3,gamee78abd094f6f5d73.log,14,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0,1,0,0,1,0,1,0,0,0,0,0,1,0,0,0,1,0,2,0,0,0,0,0,0,0,0,0,1,0,0,0,0
4,game53b40ed46697f583.log,6,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,1,0,1,1,0,0,0,0,1,0,1,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0,0,0


# Feature engineering

### Sums and averages

In [56]:
colors = ["green", "yellow", "white", "blue", "red"]

# 1. columns 1_count bis 5_count for both players combined
for num in range(1, 6):
    cols_p1 = [f"p1_{col}{num}" for col in colors]
    cols_p2 = [f"p2_{col}{num}" for col in colors]
    all_cols = cols_p1 + cols_p2
    
    df[f"{num}_count"] = df[all_cols].sum(axis=1)

# 2. Count of cards per color for both players combined
for col in colors:
    color_cols_p1 = [f"p1_{col}{num}" for num in range(1, 6)]
    color_cols_p2 = [f"p2_{col}{num}" for num in range(1, 6)]
    all_color_cols = color_cols_p1 + color_cols_p2
    
    df[f"{col}_count"] = df[all_color_cols].sum(axis=1)

# 3. Hand-sum and averages for p1 and p2
for p in ["p1", "p2"]:
    df[f"{p}_hand_sum"] = sum(
        sum(df[f"{p}_{col}{num}"] for col in colors) * num for num in range(1, 6)
    )
    df[f"{p}_avg_value"] = df[f"{p}_hand_sum"] / 5

df.head()
print(df.columns.values)
df


<ArrowStringArray>
[ 'file_source',  'final_score',    'p1_green1',    'p1_green2',
    'p1_green3',    'p1_green4',    'p1_green5',   'p1_yellow1',
   'p1_yellow2',   'p1_yellow3',   'p1_yellow4',   'p1_yellow5',
    'p1_white1',    'p1_white2',    'p1_white3',    'p1_white4',
    'p1_white5',     'p1_blue1',     'p1_blue2',     'p1_blue3',
     'p1_blue4',     'p1_blue5',      'p1_red1',      'p1_red2',
      'p1_red3',      'p1_red4',      'p1_red5',    'p2_green1',
    'p2_green2',    'p2_green3',    'p2_green4',    'p2_green5',
   'p2_yellow1',   'p2_yellow2',   'p2_yellow3',   'p2_yellow4',
   'p2_yellow5',    'p2_white1',    'p2_white2',    'p2_white3',
    'p2_white4',    'p2_white5',     'p2_blue1',     'p2_blue2',
     'p2_blue3',     'p2_blue4',     'p2_blue5',      'p2_red1',
      'p2_red2',      'p2_red3',      'p2_red4',      'p2_red5',
      '1_count',      '2_count',      '3_count',      '4_count',
      '5_count',  'green_count', 'yellow_count',  'white_count',
   'bl

,file_source,final_score,p1_green1,p1_green2,p1_green3,p1_green4,p1_green5,p1_yellow1,p1_yellow2,p1_yellow3,p1_yellow4,p1_yellow5,p1_white1,p1_white2,p1_white3,p1_white4,p1_white5,p1_blue1,p1_blue2,p1_blue3,p1_blue4,p1_blue5,p1_red1,p1_red2,p1_red3,p1_red4,p1_red5,p2_green1,p2_green2,p2_green3,p2_green4,p2_green5,p2_yellow1,p2_yellow2,p2_yellow3,p2_yellow4,p2_yellow5,p2_white1,p2_white2,p2_white3,p2_white4,p2_white5,p2_blue1,p2_blue2,p2_blue3,p2_blue4,p2_blue5,p2_red1,p2_red2,p2_red3,p2_red4,p2_red5,1_count,2_count,3_count,4_count,5_count,green_count,yellow_count,white_count,blue_count,red_count,p1_hand_sum,p1_avg_value,p2_hand_sum,p2_avg_value
0,gamec4a276e8bbb31378.log,10,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,2,1,0,1,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,0,1,2,3,4,0,1,1,1,3,4,17,3.4,13,2.6
1,game146052cdf75868b1.log,21,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,1,1,0,0,1,0,0,0,0,0,0,1,0,0,0,1,1,0,0,0,1,0,0,0,0,0,0,0,1,0,3,4,0,1,2,0,1,2,3,4,15,3.0,10,2.0
2,gamea89f449023dbd46f.log,8,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,2,0,1,0,0,0,1,0,0,0,0,0,2,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,3,2,4,0,1,1,2,1,3,3,12,2.4,12,2.4
3,gamee78abd094f6f5d73.log,14,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0,1,0,0,1,0,1,0,0,0,0,0,1,0,0,0,1,0,2,0,0,0,0,0,0,0,0,0,1,0,0,0,0,3,3,0,3,1,1,1,2,3,3,14,2.8,12,2.4
4,game53b40ed46697f583.log,6,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,1,0,1,1,0,0,0,0,1,0,1,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0,0,0,4,1,1,3,1,1,1,2,2,4,13,2.6,13,2.6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1805,gamea2ba9f1a56d55772.log,8,2,0,1,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,1,0,0,1,0,0,1,0,0,0,0,0,4,1,2,2,1,3,3,1,3,0,10,2.0,15,3.0
1806,game35435bfba505cc6c.log,7,2,0,1,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,5,2,1,1,1,5,2,1,1,1,14,2.8,7,1.4
1807,game2a50fb7f25b9d83d.log,11,2,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,1,0,2,0,0,0,0,4,3,1,1,1,3,1,1,3,2,11,2.2,11,2.2
1808,game626451c79e20ac30.log,5,2,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,2,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,2,1,3,3,1,4,3,1,1,1,12,2.4,18,3.6


### Unique cards

In [57]:
for p in ["p1", "p2"]:
    player_card_cols = []
    for col in colors:
        for num in range(1, 6):
            player_card_cols.append(f"{p}_{col}{num}")
    
    df[f"{p}_unique_cards"] = (df[player_card_cols] > 0).sum(axis=1)

df.head()

,file_source,final_score,p1_green1,p1_green2,p1_green3,p1_green4,p1_green5,p1_yellow1,p1_yellow2,p1_yellow3,p1_yellow4,p1_yellow5,p1_white1,p1_white2,p1_white3,p1_white4,p1_white5,p1_blue1,p1_blue2,p1_blue3,p1_blue4,p1_blue5,p1_red1,p1_red2,p1_red3,p1_red4,p1_red5,p2_green1,p2_green2,p2_green3,p2_green4,p2_green5,p2_yellow1,p2_yellow2,p2_yellow3,p2_yellow4,p2_yellow5,p2_white1,p2_white2,p2_white3,p2_white4,p2_white5,p2_blue1,p2_blue2,p2_blue3,p2_blue4,p2_blue5,p2_red1,p2_red2,p2_red3,p2_red4,p2_red5,1_count,2_count,3_count,4_count,5_count,green_count,yellow_count,white_count,blue_count,red_count,p1_hand_sum,p1_avg_value,p2_hand_sum,p2_avg_value,p1_unique_cards,p2_unique_cards
0,gamec4a276e8bbb31378.log,10,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,2,1,0,1,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,0,1,2,3,4,0,1,1,1,3,4,17,3.4,13,2.6,4,5
1,game146052cdf75868b1.log,21,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,1,1,0,0,1,0,0,0,0,0,0,1,0,0,0,1,1,0,0,0,1,0,0,0,0,0,0,0,1,0,3,4,0,1,2,0,1,2,3,4,15,3.0,10,2.0,5,5
2,gamea89f449023dbd46f.log,8,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,2,0,1,0,0,0,1,0,0,0,0,0,2,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,3,2,4,0,1,1,2,1,3,3,12,2.4,12,2.4,4,4
3,gamee78abd094f6f5d73.log,14,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0,1,0,0,1,0,1,0,0,0,0,0,1,0,0,0,1,0,2,0,0,0,0,0,0,0,0,0,1,0,0,0,0,3,3,0,3,1,1,1,2,3,3,14,2.8,12,2.4,4,4
4,game53b40ed46697f583.log,6,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,1,0,1,1,0,0,0,0,1,0,1,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0,0,0,4,1,1,3,1,1,1,2,2,4,13,2.6,13,2.6,5,5


### Average possible max progression per color

In [58]:
all_chains = []

for col in colors:
    # 1. Check, whether card is in p1 or p2 hand
    has_1 = (df[f"p1_{col}1"] > 0) | (df[f"p2_{col}1"] > 0)
    has_2 = (df[f"p1_{col}2"] > 0) | (df[f"p2_{col}2"] > 0)
    has_3 = (df[f"p1_{col}3"] > 0) | (df[f"p2_{col}3"] > 0)
    has_4 = (df[f"p1_{col}4"] > 0) | (df[f"p2_{col}4"] > 0)
    has_5 = (df[f"p1_{col}5"] > 0) | (df[f"p2_{col}5"] > 0)
    
    # 2. Calculate length of chain
    chain_length = (
        has_1.astype(int) + 
        (has_1 & has_2).astype(int) + 
        (has_1 & has_2 & has_3).astype(int) + 
        (has_1 & has_2 & has_3 & has_4).astype(int) + 
        (has_1 & has_2 & has_3 & has_4 & has_5).astype(int)
    )
    
    all_chains.append(chain_length)

# 3. Calculate average
df["avg_max_possible_progress"] = sum(all_chains) / len(colors)

pd.set_option('display.max_columns', None)
df.head()

,file_source,final_score,p1_green1,p1_green2,p1_green3,p1_green4,p1_green5,p1_yellow1,p1_yellow2,p1_yellow3,p1_yellow4,p1_yellow5,p1_white1,p1_white2,p1_white3,p1_white4,p1_white5,p1_blue1,p1_blue2,p1_blue3,p1_blue4,p1_blue5,p1_red1,p1_red2,p1_red3,p1_red4,p1_red5,p2_green1,p2_green2,p2_green3,p2_green4,p2_green5,p2_yellow1,p2_yellow2,p2_yellow3,p2_yellow4,p2_yellow5,p2_white1,p2_white2,p2_white3,p2_white4,p2_white5,p2_blue1,p2_blue2,p2_blue3,p2_blue4,p2_blue5,p2_red1,p2_red2,p2_red3,p2_red4,p2_red5,1_count,2_count,3_count,4_count,5_count,green_count,yellow_count,white_count,blue_count,red_count,p1_hand_sum,p1_avg_value,p2_hand_sum,p2_avg_value,p1_unique_cards,p2_unique_cards,avg_max_possible_progress
0,gamec4a276e8bbb31378.log,10,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,2,1,0,1,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,0,1,2,3,4,0,1,1,1,3,4,17,3.4,13,2.6,4,5,0.2
1,game146052cdf75868b1.log,21,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,1,1,0,0,1,0,0,0,0,0,0,1,0,0,0,1,1,0,0,0,1,0,0,0,0,0,0,0,1,0,3,4,0,1,2,0,1,2,3,4,15,3.0,10,2.0,5,5,1.2
2,gamea89f449023dbd46f.log,8,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,2,0,1,0,0,0,1,0,0,0,0,0,2,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,3,2,4,0,1,1,2,1,3,3,12,2.4,12,2.4,4,4,0.4
3,gamee78abd094f6f5d73.log,14,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0,1,0,0,1,0,1,0,0,0,0,0,1,0,0,0,1,0,2,0,0,0,0,0,0,0,0,0,1,0,0,0,0,3,3,0,3,1,1,1,2,3,3,14,2.8,12,2.4,4,4,0.6
4,game53b40ed46697f583.log,6,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,1,0,1,1,0,0,0,0,1,0,1,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0,0,0,4,1,1,3,1,1,1,2,2,4,13,2.6,13,2.6,5,5,1.4


### Save

In [59]:
# save the clean dataset as a csv file
df.to_csv("hanabi_dataset_features.csv", index=False)